# Simple RAG with Jina and PostgreSQL

Load the DispatchDesk corpus, embed its sections with `jina-embeddings-v5-text-nano`, upsert the document tables, and retrieve policy guidance through PostgreSQL/pgvector. The notebook reuses the application's corpus service, chunking strategy, embedding service, database queries, and RAG service.

Use a Python kernel with the backend dependencies installed (`uv sync --locked` from `backend`). This notebook can run from the repository root, `backend`, or this notebook folder. It needs neither Transformers nor ChromaDB.

Configure `JINA_API_KEY` and `DATABASE_URL` in `backend/.env` or the environment. The final answer cell also needs `GROQ_API_KEY`. Existing Pydantic settings load credentials at runtime; this notebook never opens `.env` directly or displays credentials. Apply migrations locally with `uv run alembic upgrade head` before running the database cells.


In [ ]:
import logging
import sys
from pathlib import Path


def find_backend_dir() -> Path:
    current = Path.cwd().resolve()
    for directory in (current, *current.parents):
        for candidate in (directory, directory / "backend"):
            if (candidate / "service" / "rag.py").is_file() and (
                candidate / "config.py"
            ).is_file():
                return candidate
    raise FileNotFoundError("Run this notebook from within the DispatchDesk repository")


BACKEND_DIR = find_backend_dir()
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from config import get_settings
from queries.vector_store import insert_chunks, retrieve
from service.corpus import CorpusService
from service.factory import create_chunking_strategy, create_embedding_service
from service.rag import answer_question

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("dispatchdesk.notebook")
CORPUS_DIR = BACKEND_DIR / "service" / "rag_data" / "corpus"
settings = get_settings()
embedding_service = create_embedding_service(settings)
corpus_service = CorpusService(
    CORPUS_DIR, BACKEND_DIR, create_chunking_strategy(settings)
)
logger.info("Backend: %s; embedding model: %s", BACKEND_DIR, embedding_service.model)

## Load and chunk the corpus

Reuse `CorpusService` with the configured chunking strategy (Markdown sections by default). It excludes the corpus README and attaches document IDs, versions, source paths, section headings, and file hashes. Loading these files makes no API request or database write.


In [ ]:
documents, ids = corpus_service.load()
if not documents:
    raise ValueError("Corpus produced no chunks")

logger.info("Loaded %s section chunks", len(documents))
for chunk_id, document in list(zip(ids, documents, strict=True))[:3]:
    logger.info("%s | %s", chunk_id, document.metadata["section"])

## Embed document passages with Jina

This cell calls the Jina API. Documents and questions use the same model and input request payload as the original pipeline. The shared embedding client logs the API-reported token usage. There is no local embedding model or tokenizer download.


In [ ]:
texts = [document.page_content for document in documents]
embeddings = embedding_service.embed_documents(texts)

if len(embeddings) != len(documents):
    raise ValueError("The number of embeddings must match the number of chunks")
logger.info(
    "Generated %s embeddings with %s dimensions", len(embeddings), len(embeddings[0])
)

## Store documents and chunks in PostgreSQL

Executing this cell writes to `app.documents` and `app.document_chunks` in the database configured by `DATABASE_URL`. It uses the existing SQLAlchemy query function, which upserts the batch in one transaction. Re-running unchanged corpus files updates their existing rows. Orders, riders, hourly metrics, and zones are not reset.


In [ ]:
stored = insert_chunks(documents=documents, embeddings=embeddings)
logger.info("Upserted %s chunks into app.documents / app.document_chunks", stored)

## Retrieve relevant guidance

Embed only the question, then use the shared PostgreSQL cosine-distance query. Retrieved passages include the chunk citation, content, and similarity. A nearest-neighbor result is a candidate source; its score alone does not prove it answers the question.


In [ ]:
question = "Why are my deliveries slipping when it rains?"
TOP_K = 3
query_embedding = embedding_service.embed_query(question)
results = retrieve(query_embedding=query_embedding, match_count=TOP_K)

if not results:
    logger.info("No guidance is stored. Run the ingestion cells first.")
for result in results:
    logger.info(
        "Source: %s | similarity: %.3f", result["chunk_id"], result["similarity"]
    )
    logger.info("%s", result["content"])

## Generate an answer through the application RAG service

This optional cell uses the same `answer_question` function as the UI, including the current system prompt and Groq model. It re-embeds the question and retrieves its own evidence; answer-generation tokens are separate from embedding tokens. It does not load or reset a scenario.


In [ ]:
answer = answer_question(question, top_k=TOP_K)
logger.info("%s", answer)